# 02 — Build the Work Opportunity Radar

This notebook follows Amina from collected evidence to a candidate-facing decision aid. It keeps three questions separate:

1. Does the post look like a current opportunity?
2. How relevant is it to this candidate?
3. What can generated text safely say from the available evidence?

It runs independently of Notebook 01 and uses the fictional offline source by default.

In [ ]:
# Local setup — safe to rerun from the project root or notebooks folder.
import json, os, sys
from pathlib import Path

def find_project(start):
    for folder in [start, *start.parents]:
        if (folder / 'config.yaml').exists() and (folder / 'src').is_dir():
            return folder
    return None

ROOT = find_project(Path.cwd().resolve())
if ROOT is None:
    raise FileNotFoundError('Open Jupyter from the work-opportunity-radar folder, then rerun.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project: {ROOT}')

## Customize me

The defaults tell Amina's story. Any candidate can replace the broad, job-relevant preferences below. Do not paste a CV, contact details, protected traits, or an API key.

In [ ]:
# CUSTOMIZE ME — edit values, not the later pipeline.
PROFILE = {
    'name': 'Amina (fictional)',
    'location': 'Nairobi, KE',
    'career_stage': 'new_graduate',
    'preferred_levels': ['internship', 'graduate', 'junior', 'entry_level'],
    'skills': ['Python', 'SQL', 'JavaScript', 'Git'],
    'target_roles': ['Junior Data Analyst', 'Graduate Software Engineer'],
    'location_preference': ['Nairobi', 'Remote'],
}
SOURCE = 'sample'  # sample, africa_ats, remotive, remoteok, csv, or json
QUERY = ''         # blank lets ranking decide relevance
PROFILE

In [ ]:
import pandas as pd
from IPython.display import display
from src import load_config, resolve
from src.collect import fetch
from src.cleaning import clean_collected

cfg = load_config()
cfg['collect']['source'] = SOURCE
cfg['collect']['query'] = QUERY
jobs = fetch(cfg, write=True, verbose=False)
trusted = clean_collected(cfg, jobs, verbose=False)
print(f'Prepared {len(trusted)} trusted records for {PROFILE["name"]}.')

## First classify the evidence — without using Amina

A classifier predicts a label; it does not verify an employer. The vocabulary deliberately keeps uncertainty visible: `likely_opportunity`, `uncertain`, and `not_current`. Notice that `PROFILE` is not passed to the classifier.

In [ ]:
from datetime import date
from src.classification import classify_frame

talk_examples = pd.DataFrame([
    {'title': 'Junior Data Analyst', 'company': 'Acacia',
     'url': 'https://example.org/apply', 'deadline': '2026-09-01',
     'description': 'We are hiring an analyst role.'},
    {'title': 'Our data team is growing', 'company': '', 'url': '', 'deadline': '',
     'description': 'Our data team is growing — DM me.'},
    {'title': 'Graduate careers event', 'company': 'Community',
     'url': 'https://example.org/event', 'deadline': '2026-06-01',
     'description': 'The career fair event has ended.'},
])
classified_examples = classify_frame(talk_examples, today=date(2026, 8, 8))
display(classified_examples[['title', 'deadline', 'opportunity_status',
                             'status_uncertainty', 'status_why']])

## Then rank relevance for this candidate

A post can look real and current but still be irrelevant. Ranking now uses the candidate's skills, target roles, preferred locations, and levels. Every score has a readable reason. Not-current records remain available separately instead of disappearing.

In [ ]:
from src.radar import build_radar

ranked, not_current = build_radar(trusted, PROFILE)
display(ranked[['fit_score', 'opportunity_status', 'title', 'company',
                'location_clean', 'why']].head(8))
print(f'Kept {len(not_current)} not-current record(s) outside the ranking.')

## Machine learning: evaluate the failure, not just the score

The public-style feed has no defensible ground-truth label, so this teaching experiment uses labelled synthetic examples. A naive random split leaks near-duplicate posts across train and test. The honest split groups duplicates together. False positives can waste time or expose a candidate to risk; false negatives can hide a useful opportunity.

In [ ]:
from src.generate_data import generate
from src.matching import run_experiment

generate(cfg)
experiment = run_experiment(cfg, verbose=False)
tn, fp, fn, tp = experiment.confusion.ravel()
display(pd.DataFrame([{
    'naive_accuracy': experiment.naive_score,
    'honest_accuracy': experiment.honest_score,
    'leakage_gap': experiment.gap,
    'false_positives': int(fp),
    'false_negatives': int(fn),
}]))

## Generated AI: plausible is not the same as true

The offline teaching stub contrasts a loose prompt that invents details with a grounded, schema-validated extraction. Unknown details must remain `null` and appear in `missing_information`.

In [ ]:
from src.application_assistant import run_demo

top_job = ranked.iloc[0].to_dict()
assistant = run_demo(cfg, verbose=False, job_record=top_job, profile=PROFILE)
print('VAGUE OUTPUT — fluent, but inspect the facts')
print(assistant['vague'])
print('\nGROUNDED OUTPUT — validated and explicit about gaps')
print(json.dumps(assistant['grounded'].model_dump(), indent=2))
print('\nDRAFT FOR HUMAN REVIEW')
print(assistant['message'])

## Optional: plug in ChatGPT without an API key

1. Run the next cell and copy the entire prompt.
2. Open ChatGPT yourself, paste the prompt, and ask it to return only JSON.
3. Copy the reply into `CHATGPT_JSON` in the validation cell.
4. Run the validation cell. Invalid or invented structure is rejected locally.

Do not paste personal contact details, a CV, or confidential information. ChatGPT output is a draft; verify consequential claims against the original source before applying.

In [ ]:
from src.application_assistant import job_context

template = resolve(cfg['paths']['grounded_prompt']).read_text(encoding='utf-8')
chatgpt_prompt = template.replace('{job_post}', job_context(top_job))
print(chatgpt_prompt)

In [ ]:
from src.application_assistant import validate_extraction

CHATGPT_JSON = r'''
'''  # Paste only ChatGPT's JSON reply between the triple quotes.

if CHATGPT_JSON.strip():
    validated_chatgpt = validate_extraction(json.loads(CHATGPT_JSON))
    print('Valid grounded response')
    print(json.dumps(validated_chatgpt.model_dump(), indent=2))
else:
    print('Optional step skipped — paste a JSON reply above when ready.')

## Finish with evidence, not instructions

The radar does not tell Amina to apply. It gives her the source, why the post may match, missing facts, uncertainty, and a verification step. Her judgement remains in the loop.

In [ ]:
from src.evidence import build_evidence_card, format_evidence_card

card = build_evidence_card(top_job, assistant['grounded'])
print(format_evidence_card(card))

## Candidate feedback closes the loop

After checking the source, a candidate can record `relevant`, `not_relevant`, or `unsure` with a reason using `python run.py feedback JOB_ID JUDGEMENT --reason "..."`. Feedback is marked for review; it does not silently become training truth.

**Try it:** Change one skill or target role in `PROFILE`, rerun the ranking, and explain which decision changed and which failure became more or less likely.